In [3]:
from google.colab import files
import pandas as pd
uploaded = files.upload()
df_raw = pd.read_csv('train.csv')
print(df_raw.head())

Saving train.csv to train (1).csv
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            3

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Step 1: Define the Problem

# Target: 'Survived' (0 = Did not survive, 1 = Survived)

# Step 2: Understand the Data

print("--- Shape ---")
print(df_raw.shape)
print("\n--- Info ---")
df_raw.info()
print("\n--- Summary Statistics ---")
print(df_raw.describe())

# Step 3: Clean the Data

df = df_raw.copy()

# 1. Drop duplicate rows (if any)
df.drop_duplicates(inplace=True)

# 2. Impute missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# 3. Create Age Groups for EDA
bins = [0, 12, 18, 35, 60, 100]
labels = ['Child (0-12)', 'Teen (13-18)', 'Young Adult (19-35)', 'Adult (36-60)', 'Senior (60+)']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

# Step 4: Generate 4-Page PDF Report
pdf_path = "titanic_eda_report.pdf"

with PdfPages(pdf_path) as pdf:

    # 1: ML Lifecycle & Missing Values Heatmap

    fig1 = plt.figure(figsize=(10, 12), tight_layout=True)
    gs1 = fig1.add_gridspec(2, 1, height_ratios=[1.2, 1])

    # Lifecycle Table
    ax_life = fig1.add_subplot(gs1[0])
    ax_life.axis('off')
    ax_life.set_title("Machine Learning Lifecycle Mapping (Titanic Task)", fontsize=14, fontweight='bold', pad=15)

    lifecycle_data = [
        ["Phase", "Step", "Description & Implementation"],
        ["Phase 1", "Problem Definition", "Binary Classification: Predict survival (0 = No, 1 = Yes)."],
        ["Phase 2", "Data Ingestion", "Acquire passenger manifests with features (Sex, Pclass, Age, Fare)."],
        ["Phase 3", "Data Cleaning", "Handle nulls in Age (median), Embarked (mode), and Cabin."],
        ["Phase 4", "EDA & Exploration", "Analyze distributions, class/gender bias, correlation matrices."],
        ["Phase 5", "Feature Engineering", "Extract Age Groups, Family Size, One-Hot Encode categories."],
        ["Phase 6", "Modeling & Eval", "Train classifiers (Logistic, Trees), evaluate ROC-AUC, F1-Score."],
        ["Phase 7", "Deployment", "Package best pipeline as REST API service for real-time inference."]
    ]
    table = ax_life.table(cellText=lifecycle_data, cellLoc='left', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.8)
    for (i, j), cell in table.get_celld().items():
        if i == 0:
            cell.set_facecolor('#1f77b4')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#f7f7f7' if i % 2 == 0 else '#ffffff')

    # Missing Values Heatmap (Raw Data)
    ax_missing = fig1.add_subplot(gs1[1])
    sns.heatmap(df_raw.isnull(), cbar=False, yticklabels=False, cmap='viridis', ax=ax_missing)
    ax_missing.set_title("Raw Dataset Missing Values Heatmap (Before Cleaning)", fontsize=12, fontweight='bold')

    pdf.savefig(fig1)
    plt.close(fig1)

    # 2: Survival Analysis by Sex & Pclass

    fig2, (ax_sex, ax_class) = plt.subplots(2, 1, figsize=(10, 12))
    fig2.suptitle("Survival Analysis: Demographics & Socioeconomic Status", fontsize=15, fontweight='bold')

    # Survival by Sex
    sns.barplot(data=df, x='Sex', y='Survived', palette='Set2', errorbar=None, ax=ax_sex)
    ax_sex.set_title("Survival Rate by Sex", fontsize=12, fontweight='bold')
    ax_sex.set_ylabel("Survival Probability")
    ax_sex.set_ylim(0, 1)
    for p in ax_sex.patches:
        ax_sex.annotate(f"{p.get_height():.2%}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                        ha='center', va='center', color='white', fontweight='bold', fontsize=12)

    # Survival by Class and Sex
    sns.barplot(data=df, x='Pclass', y='Survived', hue='Sex', palette='muted', errorbar=None, ax=ax_class)
    ax_class.set_title("Survival Rate by Passenger Class and Sex", fontsize=12, fontweight='bold')
    ax_class.set_ylabel("Survival Probability")
    ax_class.set_xlabel("Passenger Class (1 = 1st, 2 = 2nd, 3 = 3rd)")
    ax_class.set_ylim(0, 1)
    for p in ax_class.patches:
        if p.get_height() > 0:
            ax_class.annotate(f"{p.get_height():.2%}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                            ha='center', va='center', color='black', fontsize=9, fontweight='bold')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    pdf.savefig(fig2)
    plt.close(fig2)

    # 3: Survival Analysis by Age

    fig3, (ax_age_dist, ax_age_grp) = plt.subplots(2, 1, figsize=(10, 12))
    fig3.suptitle("Survival Analysis: Age Demographics", fontsize=15, fontweight='bold')

    # Age Distribution
    sns.histplot(data=df, x='Age', hue='Survived', multiple='stack', kde=True, palette='coolwarm', bins=30, ax=ax_age_dist)
    ax_age_dist.set_title("Age Distribution by Survival Status", fontsize=12, fontweight='bold')
    ax_age_dist.set_xlabel("Age")
    ax_age_dist.set_ylabel("Passenger Count")

    # Survival by Age Category
    sns.barplot(data=df, x='AgeGroup', y='Survived', palette='magma', errorbar=None, ax=ax_age_grp)
    ax_age_grp.set_title("Survival Rate by Age Category", fontsize=12, fontweight='bold')
    ax_age_grp.set_ylabel("Survival Probability")
    ax_age_grp.set_xlabel("Age Group")
    ax_age_grp.set_ylim(0, 1)
    for p in ax_age_grp.patches:
        ax_age_grp.annotate(f"{p.get_height():.2%}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                            ha='center', va='center', color='white', fontweight='bold', fontsize=11)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    pdf.savefig(fig3)
    plt.close(fig3)


    # 4: Correlation Heatmap & Fare Boxplot

    fig4, (ax_corr, ax_fare) = plt.subplots(2, 1, figsize=(10, 12))
    fig4.suptitle("Correlation Matrix & Fare Exploration", fontsize=15, fontweight='bold')

    # Correlation Matrix
    numeric_df = df.select_dtypes(include=[np.number])
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap='Blues', square=True, cbar_kws={"shrink": 0.8}, ax=ax_corr)
    ax_corr.set_title("Feature Correlation Heatmap", fontsize=12, fontweight='bold')

    # Fare vs Survival Boxplot
    sns.boxplot(data=df, x='Survived', y='Fare', palette='Set3', showfliers=False, ax=ax_fare)
    ax_fare.set_title("Fare Distribution by Survival Status (Outliers Trimmed)", fontsize=12, fontweight='bold')
    ax_fare.set_xticklabels(['Did Not Survive (0)', 'Survived (1)'])
    ax_fare.set_ylabel("Fare ($)")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    pdf.savefig(fig4)
    plt.close(fig4)

print(f"Report successfully saved as: {pdf_path}")

--- Shape ---
(891, 12)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

--- Summary Statistics ---
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.30

/tmp/ipykernel_1418/305097931.py:87: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='Sex', y='Survived', palette='Set2', errorbar=None, ax=ax_sex)
/tmp/ipykernel_1418/305097931.py:122: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='AgeGroup', y='Survived', palette='magma', errorbar=None, ax=ax_age_grp)
/tmp/ipykernel_1418/305097931.py:148: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='Survived', y='Fare', palette='Set3', showfliers=False, ax=ax_fare)
/tmp/ipykernel_1418/305097931.py:150: UserWarning: set_tick

Report successfully saved as: titanic_eda_report.pdf


In [6]:
from google.colab import files
files.download("titanic_eda_report.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>